# 自动日志指标：actor/grad_norm
verl 在训练过程中自动上报梯度范数到 wandb/TensorBoard，字段名为：

| 指标名                | 含义                    | 路径（日志中）            |
| :----------------- | :-------------------- | :----------------- |
| `actor/grad_norm`  | Actor（策略）网络的梯度 L2 范数  | `actor/grad_norm`  |
| `critic/grad_norm` | Critic（价值）网络的梯度 L2 范数 | `critic/grad_norm` |

这些指标无需手动添加，框架在每次 optimizer.step() 前后自动计算并记录。

# 梯度熔断器（Gradient Circuit Breaker）
verl 内置了异常梯度自动熔断机制：
  - "实时监控 actor/grad_norm 与 critic/grad_norm；若单步梯度范数超过阈值（默认为1000），自动标记该 step 为异常，丢弃其梯度更新" 
  
这相当于在框架层面做了一层硬保护：即使某个 batch 因数值问题导致梯度爆炸，也不会污染全局模型。

# 分布式梯度一致性校验
在多机训练中，verl 提供了跨节点梯度校验工具：

In [ ]:
from verl.utils.distributed import all_reduce_norm

# 反向传播后，聚合所有节点的梯度范数
global_grad_norms = all_reduce_norm(grad_norms)

# 检查各节点梯度差异
max_diff = max(global_grad_norms) - min(global_grad_norms)
if max_diff > 1e-5:
    log.warning(f"梯度差异过大: {max_diff}")

- "实现分布式梯度校验机制，在反向传播后同步检查各节点梯度范数" 
相关工具函数位于 verl/utils/distributed.py。

# 梯度裁剪配置
verl 的训练配置中通过 max_grad_norm 控制裁剪阈值（典型值为 1.0）：

In [ ]:
# 配置示例
actor:
  grad_norm: 1.0    # 梯度裁剪阈值

# 或启动参数中
trainer.actor_grad_norm=1.0

- 这在框架内部最终会调用 PyTorch 的 torch.nn.utils.clip_grad_norm_。

# 在 verl 源码中定位的具体建议
如果你想在 verl 的 GitHub 源码中找到精确的实现位置，建议搜索以下关键字：

| 搜索关键词                             | 预期所在文件                                                      |
| :-------------------------------- | :---------------------------------------------------------- |
| `grad_norm`                       | `verl/trainer/*.py`（如 `ppo_trainer.py` / `grpo_trainer.py`） |
| `clip_grad_norm`                  | `verl/workers/fsdp_workers.py` 或 `verl/trainer/base.py`     |
| `all_reduce_norm`                 | `verl/utils/distributed.py`                                 |
| `circuit_breaker` / `grad_norm >` | 异常处理模块，可能在 `verl/utils/` 或 trainer 目录                       |


## 典型调用链（推测）：

In [ ]:
verl/trainer/ppo/ray_trainer.py
    ↓
_update_actor() 或 _update_policy()
    ↓
loss.backward()
    ↓
torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=cfg.grad_norm)
    ↓
计算并记录 grad_norm → wandb/tensorboard
    ↓
检查是否超过熔断阈值 → 决定是否丢弃该 step

# 与其他框架的指标名对照
如果你从其他框架迁移到 verl，注意指标命名差异：

| 含义    | verl              | TRL (HuggingFace)  |
| :---- | :---------------- | :----------------- |
| 梯度范数  | `actor/grad_norm` | 嵌在 `loss/total` 中  |
| 策略熵   | `actor/entropy`   | `policy/approx_kl` |
| KL 散度 | `actor/ppo_kl`    | `objective/kl`     |

## 总结：
- verl 的梯度监测是框架内置的，你不需要自己写代码计算。只要启用了 wandb/TensorBoard 日志，就能看到 actor/grad_norm 曲线。如果你需要自定义阈值或添加额外的梯度诊断逻辑，可以在 verl/utils/distributed.py 或 trainer 的更新循环中插入钩子。